# Test tracing.py and evaluation.py

Summary of results:
- 100% in-scope skill selection and path efficiency scores (in-scope only, n=6)
- 0% markup leakage
- out-of-scope questions drag down overall skill selection and path efficiency scores
- avg latency = 100s (local qwen 7B on CPU)

Findings:
- The agent is great at its limited scope but fails at refusing scope violations
- Hallucinates tool names
- For out-of-scope cases, the final answer is right although skill_correct and path_efficient are both False

In [4]:
import sys
sys.path.insert(0, "../src")

from cx_agent.data import load_fabsa
from cx_agent.tracing import run_with_trace
from cx_agent.evaluation import run_benchmark, summarise, evaluate_record 

df_reviews, df_exploded = load_fabsa()

In [ ]:
record = run_with_trace(
    question="What are the top complaints in Banking?",
    df_reviews=df_reviews,
    df_exploded=df_exploded,
    expected_tool="describe",
    question_type="descriptive",
)

Run ID: 3a328c83
Answer: The top complaints in Banking are app-website (146 mentions), attitude-of-staff (109), ease-of-use (107), account-access (102), and general-satisfaction (76).

Metrics:


KeyError: 'metrics'

In [6]:
metrics = evaluate_record(record)

print(f"Run ID: {record['run_id']}")
print(f"Answer: {record['answer']}")
print("\nMetrics:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

Run ID: 3a328c83
Answer: The top complaints in Banking are app-website (146 mentions), attitude-of-staff (109), ease-of-use (107), account-access (102), and general-satisfaction (76).

Metrics:
  num_tool_calls: 1
  num_loops_detected: 0
  tools_used: ['describe']
  skill_correct: True
  path_efficient: True
  answer_has_markup: False
  latency_seconds: 230.66


In [7]:
# small benchmark
test_cases = [
    # Descriptive
    {"question": "What are the top complaints in Banking?",
     "expected_tool": "describe", "question_type": "descriptive"},
    {"question": "What aspects are most mentioned in Fashion?",
     "expected_tool": "describe", "question_type": "descriptive"},

    # Inferential
    {"question": "Is sentiment significantly different between app-website and speed?",
     "expected_tool": "infer", "question_type": "inferential"},
    {"question": "Compare sentiment between Banking and Fashion industries.",
     "expected_tool": "infer", "question_type": "inferential"},

    # Reporting
    {"question": "Give me a summary report for the Fashion industry.",
     "expected_tool": "report", "question_type": "reporting"},
    {"question": "Summarise feedback from Google Play.",
     "expected_tool": "report", "question_type": "reporting"},

    # Out-of-scope
    {"question": "What is the capital of France?",
     "expected_tool": None, "question_type": "out_of_scope"},
    {"question": "What is the top trending handbag with best reviews?",
     "expected_tool": None, "question_type": "out_of_scope"},
]

results = run_benchmark(test_cases, df_reviews, df_exploded)
results

[1/8] What are the top complaints in Banking?...
[2/8] What aspects are most mentioned in Fashion?...
[3/8] Is sentiment significantly different between app-website and...
[4/8] Compare sentiment between Banking and Fashion industries....
[5/8] Give me a summary report for the Fashion industry....
[6/8] Summarise feedback from Google Play....
[7/8] What is the capital of France?...
[8/8] What is the top trending handbag with best reviews?...


,question,type,expected_tool,num_tool_calls,num_loops_detected,tools_used,skill_correct,path_efficient,answer_has_markup,latency_seconds,answer
0,What are the top complaints in Banking?,descriptive,describe,1,0,[describe],True,True,False,122.51,The top complaints in Banking are related to a...
1,What aspects are most mentioned in Fashion?,descriptive,describe,1,0,[describe],True,True,False,95.13,The most mentioned aspects in Fashion are gene...
2,Is sentiment significantly different between a...,inferential,infer,1,0,[infer],True,True,False,76.95,The sentiment towards app-website is significa...
3,Compare sentiment between Banking and Fashion ...,inferential,infer,1,0,[infer],True,True,False,75.66,The sentiment analysis shows that the Fashion ...
4,Give me a summary report for the Fashion indus...,reporting,report,1,0,[report],True,True,False,116.46,The Fashion industry summary shows that 72.8% ...
5,Summarise feedback from Google Play.,reporting,report,1,0,[report],True,True,False,135.68,"The Google Play feedback includes 6,130 review..."
6,What is the capital of France?,out_of_scope,NaN,1,0,[describe],False,False,False,106.84,I can only answer questions about customer fee...
7,What is the top trending handbag with best rev...,out_of_scope,NaN,1,0,[function_name],False,False,False,75.86,I can only answer questions about customer fee...


In [8]:
summarise(results)

=== Benchmark Summary ===

Total questions:        8
Skill selection acc:    75%
Path efficiency:        75%
Markup leakage rate:    0%
Avg tool calls:         1.0
Total loops detected:   0
Avg latency (s):        100.6

=== By question type ===
              skill_correct  path_efficient  num_tool_calls  latency_seconds
type                                                                        
descriptive             1.0             1.0             1.0           108.82
inferential             1.0             1.0             1.0            76.31
out_of_scope            0.0             0.0             1.0            91.35
reporting               1.0             1.0             1.0           126.07
